# 2. Prepare and run a benchmark
Use the same settings file as notebook 1. A benchmark owns fixed event assignments;
a run owns model outputs. Reuse the benchmark ID when comparing models or seeds,
and give each experiment a new run ID. Existing stage directories are never overwritten.

Start preparation explicitly, inspect its checks, then start training. Jobs execute
in detached Python processes and log to disk. They can continue after the kernel
closes, while the computer remains awake. XGBoost is currently the implemented trainer.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from hepml.adapters.research import (
    read_settings, inspect_inputs, feature_preview, save_provenance,
    benchmark_directory, prepare_benchmark, train_benchmark,
    stage_status, check_splits, compare_runs,
)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/hepml").is_dir())
SETTINGS = Path(os.environ.get("HEPML_RESEARCH_CONFIG", REPO / "notebooks/settings.local.json"))
config = read_settings(SETTINGS)
display(pd.Series(config, name="Effective settings"))
print(f"Luminosity: {config['analysis']['lumi']:g} pb^-1 ({config['analysis']['lumi'] / 1000:g} fb^-1)")


In [ ]:
# Enable only the stage you want to start, then run its cell.
START_PREPARATION = False
START_TRAINING = False
WAIT_FOR_COMPLETION = False  # True is useful for small reproducibility/CI runs.

## Check inputs

In [ ]:
inventory, records = inspect_inputs(config)
display(inventory)
benchmark = benchmark_directory(config)
run_dir = Path(config["workspace"]) / "runs" / config["run_id"]
print("Benchmark:", benchmark)
print("Run:", run_dir)

## Prepare fixed splits
Uses the existing `hepml prepare` command, including its selection, weights and
split seed. Reads signals and backgrounds from separate directories if configured.
After completion, turn START_PREPARATION off; rerunning a completed stage is refused.

In [ ]:
if START_PREPARATION:
    display(prepare_benchmark(config, records, wait=WAIT_FOR_COMPLETION))
else:
    print("Preparation not launched. Set START_PREPARATION=True to start a new benchmark.")
display(stage_status(benchmark))

## Verify event assignments
Rerun this cell after the job finishes. Checks exact event order against the saved
assignment table, no split overlaps, and coverage of the full prepared dataset.

In [ ]:
if stage_status(benchmark)["status"] == "complete":
    display(check_splits(benchmark / "data/splits", config["mass"]))
    display(json.loads((benchmark / "data/splits" / f"split_sig{config['mass']}.meta.json").read_text()))
else:
    print("Preparation is not complete. Inspect", benchmark / "stage.log")

## Train XGBoost
Uses the saved benchmark with the study's analysis.yaml and explicit training_args
from settings. The default fitting mode is unweighted; physical weights are always
retained for evaluation. Review the luminosity and fitting policy before starting.
Training parameters, exact commands, elapsed time and output hashes are recorded.

In [ ]:
if START_TRAINING:
    display(train_benchmark(config, records, wait=WAIT_FOR_COMPLETION))
else:
    print("Training not launched. Set START_TRAINING=True after reviewing the benchmark.")
display(stage_status(run_dir))
print("Training log:", run_dir / "stage.log")

## Inspect validation results
Test scores are saved by the existing trainer, but are not displayed here.
Choose models, checkpoints and thresholds using validation. Use notebook 3 for
comparison and deliberately enable final test reporting after freezing choices.

In [ ]:
if stage_status(run_dir)["status"] == "complete":
    display(compare_runs([run_dir], split="val"))
else:
    print("Training is not complete. Rerun this cell to refresh its status.")